# Training and evaluating DistilBERT on quiz level classification with indexed answers

In [3]:
import json
import pandas as pd
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
import time
from datasets import Dataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

colab = False

if colab == True:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_quiz_indexed_runs")
else:
    data_path = "../data"
    models_path = "../models/DistilBERT_quiz_indexed_runs"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

W1014 16:31:13.634000 1648 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


### Transform data into appropriate format

Concatenate quizzes with separator token

In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_quiz(quiz):
    headlines = quiz["headlines"]

    text = f" {tokenizer.sep_token} ".join(headlines)

    return text

def quiz_transform(target_set):

    modified_set = []

    for item in target_set:
        entry = {
            "date": item["date"],
            "text": format_quiz(item),
            "label": item["answer"]
        }
        modified_set.append(entry)

    return modified_set

train_set_DB_quiz_ready = quiz_transform(train_nn_set)
validate_set_DB_quiz_ready = quiz_transform(validate_set)
test_set_DB_quiz_ready = quiz_transform(test_set)

Create Hugging Face Datasets

In [5]:
train_dataset = Dataset.from_list(train_set_DB_quiz_ready)
validate_dataset = Dataset.from_list(validate_set_DB_quiz_ready)
test_dataset = Dataset.from_list(test_set_DB_quiz_ready)

Tokenise the text

In [6]:
padding = "max_length"
truncation = True
max_length = 200

def tokenize_quiz(example):
    return tokenizer(
        example["text"],
        padding=padding,
        truncation=truncation,
        max_length=max_length
    )

train_dataset = train_dataset.map(tokenize_quiz)
validate_dataset = validate_dataset.map(tokenize_quiz)
test_dataset = test_dataset.map(tokenize_quiz)

# Size of train and test datasets
n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

### DistilBERT

Model hyperparameters and training (only run in Colab)

In [6]:
# Set the seed
seed = 42
set_seed(seed)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
num_model_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# Define training args
output_dir = str(run_dir / "checkpoints")
num_train_epochs=20
per_device_train_batch_size=8
per_device_eval_batch_size=67
eval_strategy="epoch"
# eval_steps=1
save_strategy="epoch"
# save_steps=1
disable_tqdm=True
logging_steps=1
report_to="none"
learning_rate=5e-5 # Default is 5e-5
optim="adamw_torch" # Default is "adamw_torch"
weight_decay=0 # Default is 0
warmup_steps=0 # Default is 0
lr_scheduler_type="linear" # Default is "linear"
load_best_model_at_end=True
metric_for_best_model="eval_loss"
greater_is_better=False
early_stopping_patience=3

# Assign training args
training_args = TrainingArguments(
    seed=seed,
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    eval_strategy=eval_strategy,
    # eval_steps=eval_steps,
    save_strategy=save_strategy,
    disable_tqdm=disable_tqdm,
    logging_steps=logging_steps,
    report_to=report_to,
    learning_rate=learning_rate,
    optim=optim,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    lr_scheduler_type=lr_scheduler_type,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better
)

# Early stopping option
callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)] # If "None", comment line in Trainer

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    callbacks=callbacks
)

# Train
output = trainer.train()

# Best validation loss and the step when it happened
best_step = trainer.state.best_step if hasattr(trainer.state, "best_step") else None
best_metric = trainer.state.best_metric if hasattr(trainer.state, "best_metric") else None

# Tiered run metadata
run_metadata = {
    "Model Information": {
        "Model name": model_name,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Truncation setting": truncation,
        "Padding setting": padding,
        "Max length": max_length,
    },
    "Training Hyperparameters": {
        "Number of training epochs": num_train_epochs,
        "Per-device training batch size": per_device_train_batch_size,
        "Per-device eval batch size": per_device_eval_batch_size,
        # "Number of evaluation steps": eval_steps,
        "Output directory": output_dir,
        "Save strategy": save_strategy,
        # "Save steps": save_steps,
        "Disable tqdm": disable_tqdm,
        "Logging steps": logging_steps,
        "Report to": report_to,
        "Learning rate": learning_rate,
        "Optimiser": optim,
        "Weight decay": weight_decay,
        "Warmup steps": warmup_steps,
        "Learning rate scheduler type": lr_scheduler_type,
        "Callbacks": [type(cb).__name__ for cb in callbacks],
        "Seed": seed,
        "Load best model at end": load_best_model_at_end,
        "Metric for best model": metric_for_best_model,
        "Greater is better": greater_is_better,
        "Early stopping patience": early_stopping_patience,
    },
    "Training Results": {
        "Training time and speed": output.metrics,
        "Best model checkpoint": trainer.state.best_model_checkpoint,
        "Best validation loss": trainer.state.best_metric,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '1.361', 'grad_norm': '2.67', 'learning_rate': '5e-05', 'epoch': '0.03846'}
{'loss': '1.338', 'grad_norm': '2.506', 'learning_rate': '4.99e-05', 'epoch': '0.07692'}
{'loss': '1.481', 'grad_norm': '2.927', 'learning_rate': '4.981e-05', 'epoch': '0.1154'}
{'loss': '1.496', 'grad_norm': '2.767', 'learning_rate': '4.971e-05', 'epoch': '0.1538'}
{'loss': '1.402', 'grad_norm': '2.287', 'learning_rate': '4.962e-05', 'epoch': '0.1923'}
{'loss': '1.413', 'grad_norm': '3.545', 'learning_rate': '4.952e-05', 'epoch': '0.2308'}
{'loss': '1.401', 'grad_norm': '2.125', 'learning_rate': '4.942e-05', 'epoch': '0.2692'}
{'loss': '1.329', 'grad_norm': '2.596', 'learning_rate': '4.933e-05', 'epoch': '0.3077'}
{'loss': '1.321', 'grad_norm': '2.428', 'learning_rate': '4.923e-05', 'epoch': '0.3462'}
{'loss': '1.382', 'grad_norm': '1.386', 'learning_rate': '4.913e-05', 'epoch': '0.3846'}
{'loss': '1.411', 'grad_norm': '2.26', 'learning_rate': '4.904e-05', 'epoch': '0.4231'}
{'loss': '1.373', 'grad_no

...

{'loss': '0.5136', 'grad_norm': '6.086', 'learning_rate': '4e-05', 'epoch': '4.038'}
{'loss': '0.309', 'grad_norm': '3.424', 'learning_rate': '3.99e-05', 'epoch': '4.077'}
{'loss': '0.4399', 'grad_norm': '4.065', 'learning_rate': '3.981e-05', 'epoch': '4.115'}
{'loss': '0.2362', 'grad_norm': '3.15', 'learning_rate': '3.971e-05', 'epoch': '4.154'}
{'loss': '0.2605', 'grad_norm': '3.211', 'learning_rate': '3.962e-05', 'epoch': '4.192'}
{'loss': '0.3354', 'grad_norm': '7.418', 'learning_rate': '3.952e-05', 'epoch': '4.231'}
{'loss': '0.3663', 'grad_norm': '3.623', 'learning_rate': '3.942e-05', 'epoch': '4.269'}
{'loss': '0.2219', 'grad_norm': '3.523', 'learning_rate': '3.933e-05', 'epoch': '4.308'}
{'loss': '0.4838', 'grad_norm': '7.364', 'learning_rate': '3.923e-05', 'epoch': '4.346'}
{'loss': '0.1377', 'grad_norm': '1.755', 'learning_rate': '3.913e-05', 'epoch': '4.385'}
{'loss': '0.1552', 'grad_norm': '2.16', 'learning_rate': '3.904e-05', 'epoch': '4.423'}
{'loss': '0.2325', 'grad_norm

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2097', 'grad_norm': '7.107', 'learning_rate': '3.75e-05', 'epoch': '5.038'}
{'loss': '0.1203', 'grad_norm': '3.198', 'learning_rate': '3.74e-05', 'epoch': '5.077'}
{'loss': '0.07734', 'grad_norm': '1.821', 'learning_rate': '3.731e-05', 'epoch': '5.115'}
{'loss': '0.06384', 'grad_norm': '1.514', 'learning_rate': '3.721e-05', 'epoch': '5.154'}
{'loss': '0.06734', 'grad_norm': '1.527', 'learning_rate': '3.712e-05', 'epoch': '5.192'}
{'loss': '0.2822', 'grad_norm': '6.119', 'learning_rate': '3.702e-05', 'epoch': '5.231'}
{'loss': '0.09437', 'grad_norm': '4.297', 'learning_rate': '3.692e-05', 'epoch': '5.269'}
{'loss': '0.05586', 'grad_norm': '0.8341', 'learning_rate': '3.683e-05', 'epoch': '5.308'}
{'loss': '0.05968', 'grad_norm': '1.802', 'learning_rate': '3.673e-05', 'epoch': '5.346'}
{'loss': '0.09485', 'grad_norm': '6.436', 'learning_rate': '3.663e-05', 'epoch': '5.385'}
{'loss': '0.04617', 'grad_norm': '1.743', 'learning_rate': '3.654e-05', 'epoch': '5.423'}
{'loss': '0.03

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '71.55', 'train_samples_per_second': '56.74', 'train_steps_per_second': '7.268', 'train_loss': '0.8416', 'epoch': '6'}
Run metadata saved to /content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_quiz_indexed_runs/20261014-152333/run_metadata.json


Logging and storing (only run in Colab)

In [7]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_quiz_indexed_runs/20261014-152333/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_quiz_indexed_runs/20261014-152333/best_model/tokenizer.json')

Load model for evaluation

In [7]:
model_path = Path(f"{models_path}/20261014-152333/best_model")

model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

test_inputs = {
    key: torch.tensor(test_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    outputs = model(**test_inputs)

logits = outputs.logits

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Evaluate

In [8]:
predictions = torch.argmax(logits, dim=1).numpy()
answers = np.array(test_dataset["label"])

accuracy_DistilBERT_quiz = np.mean(predictions == answers)

print(f"Quiz-level accuracy: {accuracy_DistilBERT_quiz:.2%}")

Quiz-level accuracy: 28.36%


Save results

In [9]:
# Calculate probabilites using softmax
probs = torch.softmax(logits, dim=1).cpu().numpy()

# Predicted fake headline position (0–3)
y_pred_quiz = np.argmax(probs, axis=1)

# Confidence = fake probability assigned to the selected headline
confidence = np.max(probs, axis=1)

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_DistilBERT_concat = pd.DataFrame({
    "date": test_dates,
    "prediction": y_pred_quiz,
    "confidence": confidence
})

predictions_DistilBERT_concat.to_csv(
    "../data/predictions_distilbert_concat.csv",
    index=False
)

Checking changes from old to new models

In [10]:
old = pd.read_csv(f"{data_path}/old/predictions_distilbert_concat.csv")
new = pd.read_csv(f"{data_path}/predictions_distilbert_concat.csv")

m = old[["date", "prediction"]].merge(
    new[["date", "prediction"]], on="date", how="outer",
    suffixes=("_old", "_new"), indicator=True
)

print(m["_merge"].value_counts())  # dates present in only one file
diff = m[(m["_merge"] == "both") & (m["prediction_old"] != m["prediction_new"])]
print(f"{len(diff)} of {len(m)} predictions changed")
print(diff.drop(columns="_merge").to_string(index=False))

_merge
both          67
left_only      0
right_only     0
Name: count, dtype: int64
6 of 67 predictions changed
      date  prediction_old  prediction_new
2020-12-17               3               0
2022-06-23               3               0
2023-02-09               0               3
2024-09-19               2               0
2025-02-20               3               0
2026-05-14               0               3


The accuracy after the first training run was surprisingly low (28.36%), so some investigation was undertaken to try and understand what was happening (this investigation was only undertaken for the old dataset, but the behaviour appears exactly the same with the new dataset)

The predictions are very strongly biased to indices 0 and 3

In [ ]:
predictions

array([0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 3, 0, 0, 3,
       3, 0, 0, 3, 0, 3, 3, 3, 0, 0, 3, 3, 3, 0, 0, 3, 3, 3, 3, 0, 0, 0,
       3, 0, 3, 2, 3, 0, 0, 3, 3, 3, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0,
       0])

Double checking distribution of answer indices, looks ok

In [ ]:
from collections import Counter

print("Train:", Counter(train_dataset["label"]))
print("Validate:", Counter(validate_dataset["label"]))
print("Test:", Counter(test_dataset["label"]))
print("Predictions:", Counter(predictions))

Train: Counter({1: 59, 3: 55, 0: 55, 2: 34})
Validate: Counter({3: 26, 0: 18, 1: 12, 2: 11})
Test: Counter({3: 19, 1: 18, 0: 17, 2: 13})
Predictions: Counter({np.int64(0): 41, np.int64(3): 25, np.int64(2): 1})


Checking the logit mean values to confirm it's not just noise that has led to the biased predictions

In [ ]:
logits.mean(dim=0)

tensor([ 0.3939, -0.5964, -0.2443,  0.2329])

Double checking that the texts align from initial dataset to Hugging Face dataset

In [ ]:
for i in range(5):
    original = test_set[i]
    prepared = test_dataset[i]

    expected_text = f" {tokenizer.sep_token} ".join(original["headlines"])

    print("Date:", original["date"])
    print("Text correct:", prepared["text"] == expected_text)
    print("Label correct:", prepared["label"] == original["answer"])
    print()

Date: 2019-11-23
Text correct: True
Label correct: True

Date: 2020-01-18
Text correct: True
Label correct: True

Date: 2020-02-22
Text correct: True
Label correct: True

Date: 2020-03-28
Text correct: True
Label correct: True

Date: 2020-05-09
Text correct: True
Label correct: True



Checking training accuracy, it's very high so this indicates overfitting

In [ ]:
train_inputs = {
    key: torch.tensor(train_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    train_outputs = model(**train_inputs)

train_predictions = torch.argmax(train_outputs.logits, dim=1)

print("Train predictions:", Counter(train_predictions.tolist()))
print("Train accuracy:", (train_predictions.numpy() == np.array(train_dataset["label"])).mean())

Train predictions: Counter({0: 78, 3: 69, 1: 44, 2: 12})
Train accuracy: 0.7980295566502463


Checking validation accuracy, it's similar to the test accuracy and also biases on indices 0 and 3

In [ ]:
validate_inputs = {
    key: torch.tensor(validate_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    validate_outputs = model(**validate_inputs)

validate_predictions = torch.argmax(validate_outputs.logits, dim=1)

validate_accuracy = (
    validate_predictions.numpy() == np.array(validate_dataset["label"])
).mean()

print("Validation predictions:", Counter(validate_predictions.tolist()))
print("Validation accuracy:", (validate_predictions.numpy() == np.array(validate_dataset["label"])).mean())

Validation predictions: Counter({0: 36, 3: 31})
Validation accuracy: 0.34328358208955223


Confusion matrix for validation set

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    np.array(validate_dataset["label"]),
    validate_predictions,
    labels=[0, 1, 2, 3]
)

print(cm)

[[11  0  0  7]
 [ 5  0  0  7]
 [ 6  0  0  5]
 [14  0  0 12]]


Look at model from first checkpoint

In [ ]:
model_path = Path(f"{models_path}/20260923-092608/checkpoint-26")

model_c1 = AutoModelForSequenceClassification.from_pretrained(model_path)
model_c1.eval()

test_inputs = {
    key: torch.tensor(test_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    outputs_c1 = model_c1(**test_inputs)

logits_c1 = outputs_c1.logits
predictions_c1 = torch.argmax(logits_c1, dim=1).numpy()
answers = np.array(test_dataset["label"])

accuracy_DistilBERT_quiz_c1 = np.mean(predictions == answers)

print(f"Quiz-level accuracy after first checkpoint: {accuracy_DistilBERT_quiz_c1:.2%}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1480.02it/s]


Quiz-level accuracy after first checkpoint: 26.87%


In [ ]:
logits_c1.mean(dim=0)

tensor([ 0.1112,  0.1409, -0.2116,  0.0064])

In [ ]:
predictions_c1

array([1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1])

In [ ]:
train_inputs = {
    key: torch.tensor(train_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    train_outputs_c1 = model_c1(**train_inputs)

train_predictions_c1 = torch.argmax(train_outputs_c1.logits, dim=1)

print("Train predictions after first checkpoint:", Counter(train_predictions_c1.tolist()))
print("Train accuracy after first checkpoint:", (train_predictions_c1.numpy() == np.array(train_dataset["label"])).mean())

Train predictions after first checkpoint: Counter({1: 192, 0: 11})
Train accuracy after first checkpoint: 0.3399014778325123


In [ ]:
validate_inputs = {
    key: torch.tensor(validate_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    validate_outputs_c1 = model_c1(**validate_inputs)

validate_predictions_c1 = torch.argmax(validate_outputs_c1.logits, dim=1)

print("Validation predictions:", Counter(validate_predictions_c1.tolist()))
print("Validation accuracy:", (validate_predictions_c1.numpy() == np.array(validate_dataset["label"])).mean())

Validation predictions: Counter({1: 64, 0: 3})
Validation accuracy: 0.1791044776119403


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    np.array(validate_dataset["label"]),
    validate_predictions_c1,
    labels=[0, 1, 2, 3]
)

print(cm)

[[ 0 18  0  0]
 [ 0 12  0  0]
 [ 0 11  0  0]
 [ 3 23  0  0]]


Look at model from second checkpoint

In [ ]:
model_path = Path(f"{models_path}/20260923-092608/checkpoint-52")

model_c2 = AutoModelForSequenceClassification.from_pretrained(model_path)
model_c2.eval()

test_inputs = {
    key: torch.tensor(test_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    outputs_c2 = model_c2(**test_inputs)

logits_c2 = outputs_c2.logits
predictions_c2 = torch.argmax(logits_c2, dim=1).numpy()
answers = np.array(test_dataset["label"])

accuracy_DistilBERT_quiz_c2 = np.mean(predictions_c2 == answers)

print(f"Quiz-level accuracy after second checkpoint: {accuracy_DistilBERT_quiz_c2:.2%}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 216.85it/s]


Quiz-level accuracy after second checkpoint: 28.36%


In [ ]:
logits_c2.mean(dim=0)

tensor([ 0.2786,  0.1249, -0.4473,  0.0627])

In [ ]:
predictions_c2

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0])

In [ ]:
train_inputs = {
    key: torch.tensor(train_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    train_outputs_c2 = model_c2(**train_inputs)

train_predictions_c2 = torch.argmax(train_outputs_c2.logits, dim=1)

print("Train predictions after second checkpoint:", Counter(train_predictions_c2.tolist()))
print("Train accuracy after second checkpoint:", (train_predictions_c2.numpy() == np.array(train_dataset["label"])).mean())

Train predictions after second checkpoint: Counter({0: 158, 1: 33, 3: 12})
Train accuracy after second checkpoint: 0.45320197044334976


In [ ]:
validate_inputs = {
    key: torch.tensor(validate_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    validate_outputs_c2 = model_c2(**validate_inputs)

validate_predictions_c2 = torch.argmax(validate_outputs_c2.logits, dim=1)

print("Validation predictions after second checkpoint:", Counter(validate_predictions_c2.tolist()))
print("Validation accuracy after second checkpoint:", (validate_predictions_c2.numpy() == np.array(validate_dataset["label"])).mean())

Validation predictions after second checkpoint: Counter({0: 62, 1: 5})
Validation accuracy after second checkpoint: 0.26865671641791045


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    np.array(validate_dataset["label"]),
    validate_predictions_c2,
    labels=[0, 1, 2, 3]
)

print(cm)

[[17  1  0  0]
 [11  1  0  0]
 [10  1  0  0]
 [24  2  0  0]]


It appears that there is always some collapse of prediction onto a given index, the model is likely taking too much importance from position rather than semantic meaning